In [0]:
create schema if not exists sony_dev.support_agent;

In [0]:
USE CATALOG sony_dev;
USE SCHEMA support_agent;

In [0]:
-- ============================================================
-- DATABRICKS AGENT TOOL CALLING FUNCTIONS - E-COMMERCE EXAMPLE
-- ============================================================
--create or replace catalog agent;
-- ============================================================
-- TABLE SETUP
-- ============================================================

CREATE TABLE IF NOT EXISTS orders (
    order_id        INT,
    customer_id     INT,
    product_id      INT,
    quantity        INT,
    unit_price      DOUBLE,
    order_status    STRING,  -- 'pending', 'shipped', 'delivered', 'cancelled'
    order_date      TIMESTAMP,
    estimated_delivery DATE
);

CREATE TABLE IF NOT EXISTS support_tickets (
    ticket_id       INT,
    customer_id     INT,
    order_id        INT,
    issue_type      STRING,  -- 'refund', 'shipping', 'product', 'other'
    description     STRING,
    status          STRING,  -- 'open', 'in_progress', 'resolved'
    created_at      TIMESTAMP,
    resolved_at     TIMESTAMP
);

CREATE TABLE IF NOT EXISTS inventory (
    product_id      INT,
    product_name    STRING,
    sku             STRING,
    stock_quantity  INT,
    warehouse       STRING,
    restock_date    DATE
);

INSERT INTO orders VALUES
(101, 1, 1, 2, 29.99, 'delivered',   '2024-01-10 09:00:00', '2024-01-15'),
(102, 2, 2, 1, 89.99, 'shipped',     '2024-01-12 14:00:00', '2024-01-18'),
(103, 3, 3, 1, 249.99,'pending',     '2024-01-14 11:00:00', '2024-01-20'),
(104, 4, 1, 3, 29.99, 'cancelled',   '2024-01-08 08:00:00', '2024-01-13'),
(105, 5, 2, 2, 89.99, 'delivered',   '2024-01-05 16:00:00', '2024-01-10'),
(106, 6, 3, 1, 249.99,'shipped',     '2024-01-13 10:00:00', '2024-01-19'),
(107, 7, 1, 1, 29.99, 'delivered',   '2024-01-06 12:00:00', '2024-01-11'),
(108, 8, 2, 4, 89.99, 'pending',     '2024-01-15 09:00:00', '2024-01-21'),
(109, 9, 3, 2, 249.99,'delivered',   '2024-01-03 15:00:00', '2024-01-08'),
(110, 10,1, 1, 29.99, 'shipped',     '2024-01-14 13:00:00', '2024-01-19');

INSERT INTO support_tickets VALUES
(1001, 1,  101, 'refund',    'Item arrived damaged, requesting refund.',        'open',        '2024-01-16 10:00:00', NULL),
(1002, 3,  103, 'shipping',  'Order has not moved in 3 days.',                  'in_progress', '2024-01-17 11:00:00', NULL),
(1003, 4,  104, 'refund',    'Cancelled order but no refund received yet.',     'resolved',    '2024-01-09 08:00:00', '2024-01-11 14:00:00'),
(1004, 7,  107, 'product',   'Wrong size was delivered.',                       'open',        '2024-01-12 09:00:00', NULL),
(1005, 9,  109, 'other',     'Requesting gift receipt for delivered order.',    'resolved',    '2024-01-04 10:00:00', '2024-01-05 09:00:00');

INSERT INTO inventory VALUES
(1, 'EverFit Performance T-Shirt', 'EVFT-001', 145, 'East Warehouse',  '2024-02-01'),
(2, 'AeroStride Running Shoes',    'AERS-002',  23, 'West Warehouse',  '2024-01-25'),
(3, 'Zenith Pro Smartwatch',       'ZNTH-003',   8, 'Central Warehouse','2024-02-10');


-- ============================================================
-- TOOL 1: Look up order status by order ID
-- ============================================================
CREATE OR REPLACE FUNCTION get_order_status(
  input_order_id INT COMMENT 'The order ID to look up status for'
)
RETURNS STRING
COMMENT 'Returns the current status, estimated delivery date, and total value of a given order ID.'
RETURN
  SELECT CONCAT(
    'Order ID: ',         order_id,        ', ',
    'Status: ',           order_status,    ', ',
    'Order Date: ',       CAST(order_date AS DATE), ', ',
    'Est. Delivery: ',    estimated_delivery, ', ',
    'Total Value: $',     ROUND(quantity * unit_price, 2)
  )
  FROM orders
  WHERE order_id = input_order_id
  LIMIT 1;


-- ============================================================
-- TOOL 2: Get all orders for a customer (by customer ID)
-- ============================================================
CREATE OR REPLACE FUNCTION get_customer_orders(
  input_customer_id INT COMMENT 'The customer ID whose orders should be retrieved'
)
RETURNS TABLE(
  order_id        INT,
  order_status    STRING,
  order_date      DATE,
  total_value     DOUBLE
)
COMMENT 'Returns all orders placed by a given customer ID, including status and total order value.'
RETURN
  SELECT
    order_id,
    order_status,
    CAST(order_date AS DATE) AS order_date,
    ROUND(quantity * unit_price, 2) AS total_value
  FROM orders
  WHERE customer_id = input_customer_id
  ORDER BY order_date DESC;


-- ============================================================
-- TOOL 3: Check product inventory levels
-- ============================================================
CREATE OR REPLACE FUNCTION check_inventory(
  input_product_name STRING COMMENT 'The name of the product to check inventory for'
)
RETURNS STRING
COMMENT 'Returns current stock levels, warehouse location, and restock date for a given product name.'
RETURN
  SELECT CONCAT(
    'Product: ',        product_name,   ', ',
    'SKU: ',            sku,            ', ',
    'In Stock: ',       stock_quantity, ' units, ',
    'Warehouse: ',      warehouse,      ', ',
    'Next Restock: ',   restock_date
  )
  FROM inventory
  WHERE LOWER(product_name) LIKE LOWER(CONCAT('%', input_product_name, '%'))
  LIMIT 1;

/*
-- ============================================================
-- TOOL 4: Create a support ticket
-- ============================================================
CREATE OR REPLACE FUNCTION agent.claude.create_support_ticket(
  input_customer_id INT    COMMENT 'ID of the customer raising the ticket',
  input_order_id    INT    COMMENT 'The order ID the issue is related to. Use NULL if not order-specific.',
  input_issue_type  STRING COMMENT 'Category of issue: refund, shipping, product, or other',
  input_description STRING COMMENT 'Detailed description of the issue'
)
RETURNS STRING
COMMENT 'Creates a new support ticket for a customer and returns the new ticket ID and confirmation.'
RETURN
  WITH new_ticket AS (
    INSERT INTO agent.claude.support_tickets
      (ticket_id, customer_id, order_id, issue_type, description, status, created_at)
    VALUES (
      (SELECT COALESCE(MAX(ticket_id), 1000) + 1 FROM agent.claude.support_tickets),
      input_customer_id,
      input_order_id,
      input_issue_type,
      input_description,
      'open',
      CURRENT_TIMESTAMP()
    )
  )
  SELECT CONCAT(
    'Support ticket successfully created. ',
    'Ticket ID: ', (SELECT MAX(ticket_id) FROM agent.claude.support_tickets WHERE customer_id = input_customer_id),
    '. Status: open. Our team will respond within 24 hours.'
  );
  */


-- ============================================================
-- TOOL 5: Look up existing support tickets by customer ID
-- ============================================================
CREATE OR REPLACE FUNCTION get_support_tickets(
  input_customer_id INT COMMENT 'The customer ID to retrieve support tickets for'
)
RETURNS TABLE(
  ticket_id   INT,
  order_id    INT,
  issue_type  STRING,
  status      STRING,
  description STRING,
  created_at  TIMESTAMP
)
COMMENT 'Returns all open and in-progress support tickets for a given customer ID.'
RETURN
  SELECT
    ticket_id,
    order_id,
    issue_type,
    status,
    description,
    created_at
  FROM support_tickets
  WHERE customer_id = input_customer_id
    AND status != 'resolved'
  ORDER BY created_at DESC;


-- ============================================================
-- TOOL 6: Calculate refund eligibility for an order
-- ============================================================
CREATE OR REPLACE FUNCTION check_refund_eligibility(
  input_order_id INT COMMENT 'The order ID to evaluate for refund eligibility'
)
RETURNS STRING
COMMENT 'Evaluates whether an order is eligible for a refund based on its status and delivery date. Returns eligibility status and reason.'
RETURN
  SELECT CASE
    WHEN order_status = 'cancelled' THEN
      CONCAT('Order ', input_order_id, ' is ELIGIBLE for a full refund. Reason: Order was cancelled.')
    WHEN order_status = 'delivered' AND DATEDIFF(CURRENT_DATE(), estimated_delivery) <= 30 THEN
      CONCAT('Order ', input_order_id, ' is ELIGIBLE for a refund. Reason: Delivered within the 30-day return window (',
             DATEDIFF(CURRENT_DATE(), estimated_delivery), ' days ago).')
    WHEN order_status = 'delivered' AND DATEDIFF(CURRENT_DATE(), estimated_delivery) > 30 THEN
      CONCAT('Order ', input_order_id, ' is NOT ELIGIBLE for a refund. Reason: Return window expired (',
             DATEDIFF(CURRENT_DATE(), estimated_delivery), ' days since delivery; limit is 30 days).')
    WHEN order_status IN ('pending', 'shipped') THEN
      CONCAT('Order ', input_order_id, ' is ELIGIBLE for cancellation and full refund. Reason: Order has not yet been delivered (status: ', order_status, ').')
    ELSE
      CONCAT('Order ', input_order_id, ': Unable to determine refund eligibility (status: ', order_status, ').')
  END
  FROM orders
  WHERE order_id = input_order_id
  LIMIT 1;

/*
-- ============================================================
-- TOOL 7: Product similarity search (Vector Search)
-- ============================================================
CREATE OR REPLACE FUNCTION agent.claude.search_products(
  query STRING COMMENT 'Natural language query to search for relevant products'
)
RETURNS TABLE
COMMENT 'Performs semantic similarity search over the product catalog. Returns the most relevant product descriptions for a given query.'
RETURN
  SELECT
    description AS page_content,
    id          AS metadata
  FROM vector_search(
    index      => 'agent.claude.products_vs_index',
    query      => query,
    num_results => 3
  );


-- ============================================================
-- TOOL 8: Get customer loyalty summary (aggregated stats)
-- ============================================================
CREATE OR REPLACE FUNCTION agent.claude.get_customer_loyalty_summary(
  input_customer_id INT COMMENT 'Customer ID to generate a loyalty summary for'
)
RETURNS STRING
COMMENT 'Returns a summary of a customers purchase history including total orders, lifetime spend, and most recent order status. Useful for determining loyalty tier or offering targeted discounts.'
RETURN
  SELECT CONCAT(
    'Customer ID: ',        input_customer_id,                       ', ',
    'Total Orders: ',       COUNT(*),                                ', ',
    'Delivered Orders: ',   SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END), ', ',
    'Lifetime Spend: $',    ROUND(SUM(quantity * unit_price), 2),    ', ',
    'Avg Order Value: $',   ROUND(AVG(quantity * unit_price), 2),    ', ',
    'Last Order Status: ',  FIRST_VALUE(order_status)
                              OVER (PARTITION BY customer_id ORDER BY order_date DESC)
  )
  FROM agent.claude.orders
  WHERE customer_id = input_customer_id
  GROUP BY customer_id;

  */